In [ ]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd()))
from evaluate import (
    SCORE_COLS, SECTION_LABELS,
    discover_runs, load_results, truth_coverage,
    run_evaluations,
    report_latency, report_scores,
    plot_section_heatmap, plot_s4_accuracy, plot_latency, plot_comparison,
    short_name,
)
print('evaluate loaded OK')

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────────
OUTPUT_ROOT      = Path('results')
TRUTH_ROOT       = Path('truth')
EVAL_OUTPUT_ROOT = Path('eval_output')
EVAL_CSV         = EVAL_OUTPUT_ROOT / 'eval_scores.csv'
EVAL_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

## i. Discover available results
Scan `results/` and show what runs are available, what models were used, and which context backend each run used. Use this to decide which runs and models to load and compare.

In [ ]:
# ── Overview: all runs ────────────────────────────────────────────────────────
runs_df = discover_runs(OUTPUT_ROOT)
display(runs_df)

In [ ]:
# ── Per-run detail: read config JSON for full BatchConfig ─────────────────────
import json

RUN_IDS_INSPECT = list(runs_df['run_id'])   # or narrow to specific run_ids

for run_id in RUN_IDS_INSPECT:
    cfg_files = list((OUTPUT_ROOT / run_id).glob('config_*.json'))
    if not cfg_files:
        print(f'{run_id}: no config JSON')
        continue
    cfg = json.loads(cfg_files[0].read_text())
    print(f"{'─'*60}")
    print(f"run_id      : {cfg.get('run_id')}")
    print(f"context     : {cfg.get('context', {}).get('type')}  "
          f"{cfg.get('context', {})}")
    print(f"models      : {[m['name'] if isinstance(m, dict) else m for m in cfg.get('models', [])]}")
    print(f"plot_filter : {cfg.get('plot_filter')}")
    print(f"ref_dir     : {cfg.get('ref_dir')}")
    print(f"run_metadata: {cfg.get('run_metadata')}")

## ii. Load and summarize
Select run IDs to load, then inspect response counts, model coverage, and latency. No judge call needed for this section.

In [ ]:
# ── Select run IDs and load ───────────────────────────────────────────────────
RUN_IDS = ['localRAG', 'YAML']   # pick from discover output above
MODELS  = None                    # None = all models found; or ['google/gemma4-31b', ...]

df = load_results(OUTPUT_ROOT, RUN_IDS, models=MODELS)

In [ ]:
# ── Coverage: run_id × model × plot ──────────────────────────────────────────
print('=== Response count by run_id × model ===')
display(
    df.groupby(['run_id', 'model'])['image_name']
    .count().rename('n_responses')
    .unstack('model')
    .fillna(0).astype(int)
)

print('\n=== Plots covered per run_id ===')
display(
    df.groupby('run_id')['plot_name']
    .apply(lambda s: ', '.join(sorted(s.unique())))
    .rename('plots')
)

In [ ]:
# ── Truth coverage: which images have ground-truth files? ─────────────────────
cov = truth_coverage(df, TRUTH_ROOT)
display(cov)

In [ ]:
# ── Latency report (from raw responses, no judge needed) ──────────────────────
report_latency(df, group_col='model')

In [ ]:
# ── Error check ───────────────────────────────────────────────────────────────
errors = df[df['error'].notna()]
if errors.empty:
    print('No errors.')
else:
    print(f'{len(errors)} errors:')
    display(errors[['run_id', 'model', 'plot_name', 'image_name', 'error']])

## iii. Configure and run evaluation
Choose a judge model and which models to evaluate. Results are cached in `EVAL_CSV` — interrupted runs resume automatically.

In [ ]:
# ── Judge configuration ───────────────────────────────────────────────────────
# Judge model: should be different from the evaluated models.
# Use list_models() from owui_client to check availability.
JUDGE_MODEL = 'vllm.gpt-oss:120b'
DELAY       = 1.0   # seconds between judge calls

# Map each evaluated model to one or more judge models.
# Models not in this dict are skipped.
JUDGE_MODEL_FOR = {m: [JUDGE_MODEL] for m in df['model'].unique()}
print('Judge config:')
for m, js in JUDGE_MODEL_FOR.items():
    print(f'  {short_name(m):20s} → {js}')

In [ ]:
# ── Run evaluations (cached) ──────────────────────────────────────────────────
df_eval = run_evaluations(
    df, TRUTH_ROOT, JUDGE_MODEL_FOR, EVAL_CSV, delay=DELAY,
)

In [ ]:
# ── Score report ──────────────────────────────────────────────────────────────
# PRIMARY_AXIS controls how the report groups results:
#   'model'  → compare models within each run_id
#   'run_id' → compare context backends within each model
PRIMARY_AXIS = 'model'

report_scores(df_eval, group_col=PRIMARY_AXIS)

In [ ]:
# ── Browse individual responses ───────────────────────────────────────────────
# Useful for diagnosing specific failures before looking at aggregate plots.
BROWSE_RUN_ID = RUN_IDS[0]
BROWSE_MODEL  = df['model'].iloc[0]
BROWSE_PLOT   = df['plot_name'].iloc[0]

subset = df[
    (df['run_id']    == BROWSE_RUN_ID) &
    (df['model']     == BROWSE_MODEL)  &
    (df['plot_name'] == BROWSE_PLOT)
]
for _, row in subset.iterrows():
    print('=' * 72)
    print(f"run_id  : {row['run_id']}")
    print(f"model   : {row['model']}")
    print(f"image   : {row['image_name']}")
    print(f"latency : {row['latency_s']}s")
    print()
    if row['error']:
        print(f"ERROR: {row['error']}")
    else:
        print(row['response'])
    print()

## iv. Comparison plots
Configure the comparison axis. **`COMPARE_COL`** is the dimension you want to compare (e.g. `'run_id'` to compare context backends, `'model'` to compare models). **`ROW_COL`** is the per-subplot grouping (the other axis).

Any column present in `df_eval` can be used — `run_id`, `model`, `plot_name`, `judge_model`.

In [ ]:
# ── Comparison configuration ──────────────────────────────────────────────────
COMPARE_COL = 'run_id'    # dimension to compare  (e.g. context backend)
ROW_COL     = 'model'     # subplot grouping       (e.g. model)

print(f'Comparing {COMPARE_COL} values: {sorted(df_eval[COMPARE_COL].unique())}')
print(f'One subplot per {ROW_COL}:      {sorted(df_eval[ROW_COL].unique())}')

In [ ]:
# ── Section score heatmap — one per COMPARE_COL value ─────────────────────────
for val in sorted(df_eval[COMPARE_COL].unique()):
    sub = df_eval[df_eval[COMPARE_COL] == val]
    plot_section_heatmap(
        sub, ROW_COL,
        title=f'Section scores  |  {COMPARE_COL}={val}',
        out_path=EVAL_OUTPUT_ROOT / f'section_heatmap_{COMPARE_COL}_{val}.png',
    )

In [ ]:
# ── S4 decision accuracy ───────────────────────────────────────────────────────
for val in sorted(df_eval[COMPARE_COL].unique()):
    sub = df_eval[df_eval[COMPARE_COL] == val]
    plot_s4_accuracy(
        sub, ROW_COL,
        title=f'S4 decision accuracy  |  {COMPARE_COL}={val}',
        out_path=EVAL_OUTPUT_ROOT / f's4_accuracy_{COMPARE_COL}_{val}.png',
    )

In [ ]:
# ── Latency ───────────────────────────────────────────────────────────────────
plot_latency(
    df, ROW_COL,
    title='Generation latency by model',
    out_path=EVAL_OUTPUT_ROOT / 'latency.png',
)

In [ ]:
# ── Side-by-side comparison: COMPARE_COL values, one subplot per ROW_COL ──────
plot_comparison(
    df_eval,
    compare_col=COMPARE_COL,
    row_col=ROW_COL,
    title=f'Score comparison: {COMPARE_COL}  (one subplot per {ROW_COL})',
    out_path=EVAL_OUTPUT_ROOT / f'comparison_{COMPARE_COL}_by_{ROW_COL}.png',
)

# Flip axes for the other direction:
plot_comparison(
    df_eval,
    compare_col=ROW_COL,
    row_col=COMPARE_COL,
    title=f'Score comparison: {ROW_COL}  (one subplot per {COMPARE_COL})',
    out_path=EVAL_OUTPUT_ROOT / f'comparison_{ROW_COL}_by_{COMPARE_COL}.png',
)

In [ ]:
# ── Per-plot-type breakdown ────────────────────────────────────────────────────
# Useful when multiple plot types are in the eval set.
for plot_name in sorted(df_eval['plot_name'].unique()):
    sub = df_eval[df_eval['plot_name'] == plot_name]
    plot_section_heatmap(
        sub, ROW_COL,
        title=f'Section scores  |  {plot_name}',
        out_path=EVAL_OUTPUT_ROOT / 'per_plot' / f'section_{plot_name}.png',
    )